In [ ]:
from _static.common.helpers import setup_hardware_client

In [ ]:
setup_hardware_client()

In [ ]:
from torch.nn import Identity
from torch.utils.data import DataLoader

import hxtorch
import hxtorch.snn as hxsnn
from hxtorch.snn.transforms import weight_transforms

from dlens_vx_v3 import halco

from tqdm.auto import tqdm
from functools import partial

In [ ]:
"""
SHD Dataset class
"""
import os
from typing import Callable, Tuple, Any, Optional
import torch
from torch.utils.data.dataset import Dataset
from torchvision.datasets.mnist import download_and_extract_archive
import h5py
import numpy as np


class SHD(Dataset):
    """ SHD dataset """

    resources = [("https://compneuro.net/datasets/shd_test.h5.gz",
                  "3062a80ec0c5719404d5b02e166543b1"),
                 ("https://compneuro.net/datasets/shd_train.h5.gz",
                  "d47c9825dee33347913e8ce0f2be08b0")]

    training_file = 'shd_train.h5'
    test_file = 'shd_test.h5'

    def __init__(self,
                 root: str = "./",
                 train: bool = True,
                 download: bool = False,
                 transform: Optional[Callable] = None,
                 device: Optional[torch.device] = None) -> None:
        super().__init__()
        self.root = root
        self.train = train
        self.transform = transform
        if device:
            self.device = device
        else:
            self.device = torch.get_default_device()

        if download:
            self.download()

        if not self._check_exists():
            raise RuntimeError('Dataset not found.'
                               + 'You can use download=True to download it')

        if self.train:
            data_file = self.training_file
        else:
            data_file = self.test_file
        self.data = h5py.File(os.path.join(self.root, data_file))
        self.times = self.data["spikes/times"]
        self.units = self.data["spikes/units"]
        self.labels = self.data["labels"]
        self.keys = self.data["extra/keys"]

    def _check_exists(self) -> bool:
        return (os.path.exists(os.path.join(self.root, self.training_file))
                and os.path.exists(os.path.join(self.root, self.test_file)))

    def get_max_time(self) -> float:
        """ Get time of latest spike present in the dataset. """
        max_value = -1
        for index in range(len(self)):
            cur_max = self.times[index][-1]
            if cur_max > max_value: 
                max_value = cur_max
        return max_value

    def get_number_of_channels(self) -> int:
        """ Get the total number of channels present in the dataset. """
        max_value = -1
        for index in range(len(self)):
            cur_max = np.max(self.units[index])
            if cur_max > max_value:
                max_value = cur_max
        return max_value + 1

    def __getitem__(self, index: int) -> Tuple[Any, Any]:
        """
        Args:
          index (int): Index
        Returns:
          tuple: (image, labels) where labels is index of the label class.
        """

        spikes = torch.stack([
            torch.from_numpy(self.times[index].astype(np.float16)).\
                to(self.device),
            torch.from_numpy(self.units[index].astype(np.float16)).\
                to(self.device)]).T
        label = self.labels[index]
        if self.transform is not None:
            spikes = self.transform(spikes)

        return spikes, int(label)

    def __len__(self) -> int:
        return len(self.labels)

    @property
    def classes(self) -> str:
        return list(self.keys[:])

    def download(self):
        """Download SHD data if it doesn't exist in given root already."""

        if self._check_exists():
            return

        os.makedirs(self.root, exist_ok=True)

        # download files
        for url, md5 in self.resources:
            filename = url.rpartition('/')[2]
            download_and_extract_archive(url,
                                         download_root=self.root,
                                         filename=filename,
                                         md5=md5)


In [ ]:
class SNN(torch.nn.Module):
    """ SNN with one hidden LIF layer and one readout LI layer """

    def __init__(
            self,
            n_in: int,
            n_hidden: int,
            n_out: int,
            mock: bool,
            dt: float,
            tau_mem: float,
            tau_syn: float,
            alpha: float,
            trace_shift_hidden: int,
            trace_shift_out: int,
            weight_init_hidden: Tuple[float, float],
            weight_init_output: Tuple[float, float],
            weight_scale: float,
            trace_scale: float,
            input_repetitions: int,
            device: torch.device):
        """
        :param n_in: Number of input units.
        :param n_hidden: Number of hidden units.
        :param n_out: Number of output units.
        :param mock: Indicating whether to train in software or on hardware.
        :param dt: Time-binning width.
        :param tau_mem: Membrane time constant.
        :param tau_syn: Synaptic time constant.
        :param trace_shift_hidden: Indicates how many indices the membrane
            trace of hidden layer is shifted to left along time axis.
        :param trace_shift_out: Indicates how many indices the membrane
            trace of readout layer is shifted to left along time axis.
        :param weight_init_hidden: Hidden layer weight initialization mean
            and std value.
        :param weight_init_output: Output layer weight initialization mean
            and std value.
        :param weight_scale: The factor with which the software weights are
            scaled when mapped to hardware.
        :param input_repetitions: Number of times to repeat input channels.
        :param device: The used PyTorch device used for tensor operations in
            software.
        """
        super().__init__()

        self.dt = dt

        # Instance to work on
        self.n_in = n_in
        self.n_hidden = n_hidden
        self.tau_mem = tau_mem
        self.tau_syn = tau_syn
        self.input_repetitions = input_repetitions
        self.weight_scale = weight_scale
        self.trace_scale = trace_scale
        self.trace_shift_hidden = trace_shift_hidden

        # Experiment instance
        self.experiment = hxsnn.Experiment(mock=mock, dt=dt)
        # To avoid time-consuming implicit calibration from given parameters
        # we load a prepared calibration. Note, that changing "Neuron"
        # hardware parameters become ineffective
        #save_nightly_calibration('spiking2_calix-native.pkl')
        #self.experiment.default_execution_instance.load_calib(
        #    "spiking2_calix-native.pkl")

        # Repeat input
        self.input_repetitions = input_repetitions

        # Input projection
        self.linear_h = hxsnn.Synapse(
            n_in * input_repetitions,
            n_hidden,
            experiment=self.experiment,
            transform=partial(
                weight_transforms.linear_saturating, scale=weight_scale))

        # Initialize weights
        if weight_init_hidden:
            w = torch.zeros(n_hidden, n_in)
            torch.nn.init.normal_(w, *weight_init_hidden)
            self.linear_h.weight.data = w.repeat(1, input_repetitions)

        # Hidden layer
        self.lif_h = hxsnn.LIF(
            n_hidden,
            experiment=self.experiment,
            tau_mem=tau_mem,
            tau_syn=tau_syn,
            leak=hxsnn.MixedHXModelParameter(0., 80),
            reset=hxsnn.MixedHXModelParameter(0., 80),
            threshold=hxsnn.MixedHXModelParameter(1., 150),
            i_synin_gm=500,
            synapse_dac_bias=1000,
            trace_scale=trace_scale,
            cadc_time_shift=trace_shift_hidden,
            shift_cadc_to_first=True)

        # Output projection
        self.linear_o = hxsnn.Synapse(
            n_hidden,
            n_out,
            experiment=self.experiment,
            transform=partial(
                weight_transforms.linear_saturating, scale=weight_scale))

        # Readout layer
        self.li_readout = hxsnn.LI(
            n_out,
            experiment=self.experiment,
            tau_mem=tau_mem,
            tau_syn=tau_syn,
            leak=hxsnn.MixedHXModelParameter(0., 80),
            i_synin_gm=500,
            synapse_dac_bias=1000,
            trace_scale=trace_scale,
            cadc_time_shift=trace_shift_out,
            shift_cadc_to_first=True,
            placement_constraint=list(
                halco.LogicalNeuronOnDLS(
                    hxsnn.morphology.SingleCompartmentNeuron(1).compartments,
                    halco.AtomicNeuronOnDLS(
                        halco.NeuronRowOnDLS(1), halco.NeuronColumnOnDLS(nrn)))
                for nrn in range(n_out)))

        # Initialize weights
        if weight_init_output:
            torch.nn.init.normal_(self.linear_o.weight, *weight_init_output)

        # Device
        self.device = device
        self.to(device)

    def forward(self, spikes: torch.Tensor) -> torch.Tensor:
        """
        Perform a forward path.
        :param spikes: LIFObservables holding spikes as input.
        :return: Returns the output of the network, i.e. membrane traces of the
        readout neurons.
        """
        # Remember input spikes for plotting
        self.s_in = spikes
        spikes = spikes.squeeze(0)
        # Transpose to (T, N)
        spikes = spikes.t()
        spikes = spikes.unsqueeze(1)
        # Increase synapse strength by repeating each input
        spikes = spikes.repeat(1, self.input_repetitions, 1)
        # Spike input handle
        print("WA")
        spikes_handle = hxsnn.LIFObservables(spikes=spikes)
        print("WA1")
        # Forward
        c_h = self.linear_h(spikes_handle)
        print("WA2")
        self.s_h = self.lif_h(c_h)  # Keep spikes for fire reg.
        print("WA3")
        c_o = self.linear_o(self.s_h)
        print("WA4")
        self.y_o = self.li_readout(c_o)
        print("WA5")
        # Execute on hardware
        hxtorch.snn.run(self.experiment, spikes.shape[0])
        print("WA6")

        return self.y_o.membrane_cadc

    def regularize(self, reg_readout):
        return reg_readout * self.linear_o.weight.norm(p=2)**2

In [ ]:
class Model(torch.nn.Module):
    """
    Model combining encoder, network, and decoder.
    """

    def __init__(self, network, decoder, readout_scale):
        super(Model, self).__init__()
        self.network = network
        self.decoder = decoder
        self.readout_scale = readout_scale

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """
        Perform forward pass through whole model, i.e.
        data -> encoder -> network -> decoder -> output
        :param inputs: tensor input data
        :returns: Returns tensor output
        """
        #spikes = self.encoder(inputs)
        spikes = inputs.squeeze(0)  # Remove batch dim since model doesn't handle batches
        print("KE", spikes, spikes.shape)
        traces = self.network(spikes)
        print("KE1")
        self.scores = self.decoder(traces).clone()
        print("KE2")
        # scale outputs
        with torch.no_grad():
            self.scores *= self.readout_scale
        print("KE3")
        return self.scores

    def regularize(self, reg_readout):
        return self.network.regularize(reg_readout)

In [ ]:
"""class IdentityLayer(hxtorch.perceptron.nn.Layer):
    def __init__(self):
        super(IdentityLayer, self).__init__()

    def forward(self, x):
        return x

encoder = IdentityLayer()"""

In [ ]:
def predict(model, data, target, loss_func):
    """ """
    print(data.shape)
    print(model)
    spikes = data  # data is already the spike matrix from transform
    scores = model(spikes.to(device))    
    loss = model.network.regularize(reg_readout=0.0004)
    loss = loss_func(scores, target) + loss
    return scores, loss


def stats(model, scores, target):
    """ """
    # Train accuracy
    pred = scores.cpu().argmax(dim=1)
    acc = pred.eq(target.view_as(pred)).float().mean().item()
    # Firing rates
    rate = model.network.s_h.spikes.sum().item() / scores.shape[0]
    return acc, rate


def train(model: torch.nn.Module,
          loader: DataLoader,
          loss_func: torch.nn.CrossEntropyLoss,
          optimizer: torch.optim.Optimizer,
          epoch: int, update):
    """
    Perform training for one epoch.
    :param model: The model to train.
    :param loader: Pytorch DataLoader instance providing training data.
    :param optimizer: The optimizer used or weight optimization.
    :param epoch: Current epoch for logging.
    :returns: Tuple (training loss, training accuracy)
    """
    model.train()
    loss, acc = 0., 0.
    n_total = len(loader)

    pbar = tqdm(total=len(loader), unit="batch", leave=False)
    for data, target in loader:

        model.zero_grad()

        scores, loss_b = predict(model, data.to(device), target.to(device), loss_func)

        loss_b.backward()
        optimizer.step()

        acc_b, rate_b = stats(model, scores, target)

        acc += acc_b / n_total
        loss += loss_b.item() / n_total

        update(n_total, loss_b.item(), 100 * acc_b, rate_b)

        pbar.set_postfix(
            epoch=f"{epoch}", loss=f"{loss_b.item():.4f}", acc=f"{acc_b:.4f}",
            rate=f"{rate_b:.2f}", lr=f"{optimizer.param_groups[-1]['lr']}")
        pbar.update()
    pbar.close()

    return loss, acc


def test(model: torch.nn.Module,
         loader: torch.utils.data.DataLoader,
         loss_func: torch.nn.CrossEntropyLoss,
         epoch: int, update):
    """
    Test the model.
    :param model: The model to test
    :param loader: Data loader containing the test data set
    :param epoch: Current trainings epoch.
    :returns: Tuple of (test loss, test accuracy)
    """
    model.eval()
    dev = model.network.device

    loss, acc, rate = 0., 0., 0
    data, target, scores = [], [], []
    n_total = len(loader)

    pbar = tqdm(total=len(loader), unit="batch", leave=False)
    for data_b, target_b in loader:
        print(f"Data shape: {data_b.to(device).shape}")
        print(f"Target shape: {target_b.to(device).shape}")
        scores_b, loss_b = predict(model, data_b.to(device), target_b.to(device), loss_func)
        scores.append(scores_b.detach())
        data.append(data_b.detach())
        target.append(target_b.detach())

        acc_b, rate_b = stats(model, scores_b, target_b)
        acc += acc_b / n_total
        loss += loss_b.item() / n_total
        rate += rate_b / n_total

        pbar.update()
    pbar.close()
    print(f"Test epoch: {epoch}, average loss: {loss:.4f}, test acc={100 * acc:.2f}%")

    scores = torch.stack(scores).reshape(-1, 20).cpu()
    data = torch.stack(data).reshape(-1, 700*100).cpu()
    target = torch.stack(target).reshape(-1).cpu()

    update(
        model.network.s_in.detach().cpu(),
        model.network.s_h.spikes.detach().cpu(),
        model.network.y_o.membrane_cadc.detach().cpu(),
        data, target, scores,
        loss, 100 * acc, rate)

    return loss, acc, rate

In [ ]:
def update(*args, **kwargs):
    """Update function that accepts variable arguments for logging/plotting"""
    return

In [ ]:
class TransformSpikesToTimeWindows:
    def __init__(self, nb_steps, max_time, n_channels):
        self.nb_steps = nb_steps
        self.n_channels = n_channels
        self.time_bins = np.linspace(0, max_time, num=nb_steps)

    def __call__(self, sample):
        times = sample[:, 0].numpy()
        channels = sample[:, 1].long().numpy()

        bins = np.digitize(times, self.time_bins)

        spike_matrix = np.zeros((self.n_channels, self.nb_steps))

        for t_bin, ch in zip(bins, channels):
            if t_bin < self.nb_steps:
                spike_matrix[ch, t_bin] += 1

        return torch.tensor(spike_matrix, dtype=torch.float32)

In [ ]:
max_time = 1.4
nb_steps = 100
n_channels = 700

In [ ]:
def events_to_spike_matrix(events, n_in=700, nb_steps=100, max_time=1.0):
    # events: (N, 2) → [time, neuron_id]

    times = events[:, 0]
    neurons = events[:, 1].long()

    time_bins = torch.linspace(0, max_time, steps=nb_steps)

    spike_matrix = torch.zeros(nb_steps, n_in)

    bins = torch.bucketize(times, time_bins)

    for t_bin, neuron in zip(bins, neurons):
        if t_bin < nb_steps and neuron < n_in:
            spike_matrix[t_bin, neuron] += 1

    return spike_matrix

In [ ]:
def batch_events_to_spike_matrix(batch_events, n_in=700, nb_steps=100, max_time=1.0):
    batch_size = batch_events.shape[0]
    spike_batch = []

    for events in batch_events:
        spike_matrix = events_to_spike_matrix(events, n_in, nb_steps, max_time)
        spike_batch.append(spike_matrix)

    return torch.stack(spike_batch)  # (B, T, N)

In [ ]:
# spikes = batch_events_to_spike_matrix(data)

In [ ]:
transform_matrix = TransformSpikesToTimeWindows(
    nb_steps=nb_steps,
    max_time=max_time,
    n_channels=700
)

In [ ]:
N_HIDDEN      = 2
MOCK          = True
DT            = 2.0e-06  # s

# We need to specify the device we want to use on the host computer
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# The SNN
snn = SNN(
    n_in=700,
    n_hidden=N_HIDDEN,
    n_out=20,
    mock=MOCK,
    dt=DT,
    tau_mem=6.0e-06,
    tau_syn=6.0e-06,
    alpha=50.,
    trace_shift_hidden=int(.0e-06/DT),
    trace_shift_out=int(.0e-06/DT),
    weight_init_hidden=(0.001, 0.25),
    weight_init_output=(0.0, 0.1),
    weight_scale=66.39,
    trace_scale=0.0147,
    input_repetitions=1 if MOCK else 5,
    device=device)

# print("Before conversion:", data.shape)
# spikes = events_to_spike_matrix(data)
# print("After conversion:", spikes.shape)

# Create Model
from hxtorch.snn.transforms.decode import MaxOverTime
decoder = MaxOverTime()

model = Model(snn, decoder, readout_scale=10.)
model

In [ ]:
# Training params
LR            = 0.002
STEP_SIZE     = 5
GAMMA         = 0.9
EPOCHS        = 4 # Adjust here for longer training...
BATCH_SIZE    = 1  # Changed to 1 since model doesn't handle batches
TRAINSET_SIZE = 2
TESTSET_SIZE  = 2

In [ ]:
# PyTorch stuff... optimizer, scheduler and loss like you normally do.
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)
loss = torch.nn.CrossEntropyLoss()

# Data loaders
train_data = SHD(
    root="./",
    train=True,
    transform=transform_matrix,
    device=device
)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

print("train data:", train_data)

test_data = SHD(
    root="./",
    train=False,
    transform=transform_matrix,
    device=device
)
test_loader  = DataLoader(test_data, batch_size=BATCH_SIZE)

# print("Batch shape:", data_b.shape)

In [ ]:
print(f"Train data length: {len(train_data)}")

In [ ]:
# Initialize the hardware
if not MOCK:
    hxtorch.init_hardware()

# Train and test
for epoch in range(0, EPOCHS + 1):
    # Test
    print(f"Test at epoch: {epoch}")
    loss_test, acc_test, rate_test = test(
        model, test_loader, loss, epoch, update)

    # Train epoch
    if epoch < EPOCHS:
        print(f"Train at epoch: {epoch}")
        loss_train, acc_train = train(
            model, train_loader, loss, optimizer, epoch, update)

    scheduler.step()

# Release the hardware connection
hxtorch.release_hardware()